# ADAPT â€” Zero-Shot Domain Transfer Training

Trains **ADAPT** (primary) + **AMAN** + **DMAN** (support roles) with GRPO.

**ADAPT** learns to map unknown scheduling domains (e.g. Hospital ICU) to ATC parameters using structural reasoning alone â€” no domain keywords, just numbers.

**Training mix:** ~50% ADAPT, ~25% AMAN, ~25% DMAN  
**Method:** 4-bit QLoRA (rank=32, all 7 projection types) + GRPO  
**Recommended runtime:** `Runtime â†’ Change runtime type â†’ GPU (T4)`

## 1. Configuration

In [ ]:
from pathlib import Path
import os

REPO_URL   = "https://github.com/GTsingh600/adapt-atc-final.git"
REPO_DIR   = Path("/content/ATC")
USE_DRIVE  = True
OUTPUT_DIR = (
    Path("/content/drive/MyDrive/atc-adapt")
    if USE_DRIVE else Path("/content/atc-adapt")
)

MODEL_NAME    = "Qwen/Qwen2.5-1.5B-Instruct"  # 1.5B fits T4; use 7B on A100
EPISODES      = 200    # 30-ep SFT + 200-ep GRPO is the full pipeline
LORA_RANK     = 8      # rank-8 for 1.5B — ~0.4% trainable, T4-safe
N_GENERATIONS = 2      # GRPO group size (2 = T4 safe)
SEED          = 42
EVAL_EPISODES = 5

os.environ["WANDB_MODE"]             = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"]       = "1"

print(f"MODEL_NAME    = {MODEL_NAME}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")
print(f"EPISODES      = {EPISODES}")
print(f"LORA_RANK     = {LORA_RANK}  (all 7 projection types)")
print(f"N_GENERATIONS = {N_GENERATIONS}")
print(f"Focus         = ADAPT domain transfer (~65% of samples)")

## 2. Mount Drive and clone repo

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output ready: {OUTPUT_DIR}")

In [ ]:
import shutil, subprocess, sys, os
from pathlib import Path

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

## 3. Install dependencies

In [ ]:
import subprocess, sys

def pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip("-U", "pip")
pip(
    "flash-attn>=2.5.0",
    "trl>=0.16.0",
    "transformers>=4.47.0",
    "huggingface_hub>=0.24.0",  # trl<0.16 broke on <0.24
    "datasets>=2.20.0",
    "accelerate>=0.32.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.0",
    "matplotlib>=3.9.0",
    "numpy>=1.26.0",
    "openenv-core[core]>=0.2.3",
    "fastapi>=0.111.0",
    "openai>=1.30.0",
    "pydantic>=2.7.0",
    "uvicorn>=0.29.0",
)
print("Done.")

## 4. Sanity check â€” verify ADAPT-focused dataset

In [ ]:
import torch
from training.dataset import build_episode_dataset

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:  {props.name}  |  VRAM: {props.total_memory/1e9:.1f} GB")

sample = build_episode_dataset(
    n_episodes=4, seed=SEED,
    include_adapt=True,
    domain_episode_ratio=0.65, long_horizon_ratio=0.0,
)
roles = sorted({r["agent_role"] for r in sample})
print(f"Samples: {len(sample)}  |  Roles: {roles}")
assert "ADAPT" in roles, "ADAPT missing"
    "assert roles == ['ADAPT', 'AMAN', 'DMAN'], f'Expected 3 roles, got {roles}'\n",
print(f"3 roles present — ADAPT ~65% primary, AMAN+DMAN ~35% micro support")

## 4b. Phase 1 — SFT Warmup (~10 min)

Teach the 1.5B model correct JSON output format **before** GRPO starts.  
Uses micro tasks (5-6 flights) for AMAN/DMAN + ICU domain for ADAPT.  
65% of samples are ADAPT (simple ~120-token output) — warms up the primary role first.

> Skip this cell if resuming from an existing SFT checkpoint.

In [ ]:
import sys, os
from pathlib import Path
from training.train_sft import train_sft

_g = globals()
MODEL_NAME = _g.get('MODEL_NAME', 'Qwen/Qwen2.5-1.5B-Instruct')
OUTPUT_DIR = Path(_g.get('OUTPUT_DIR', '/content/atc-adapt'))
SEED       = _g.get('SEED', 42)

SFT_DIR = OUTPUT_DIR / 'sft-warmup'

sft_checkpoint = train_sft(
    model_name  = MODEL_NAME,
    output_dir  = str(SFT_DIR),
    n_episodes  = 30,      # ~75 samples, ~10 min on T4
    lora_rank   = 8,       # match GRPO rank
    seed        = SEED,
    num_epochs  = 3,
)

# Override MODEL_NAME so GRPO cell starts from SFT checkpoint
MODEL_NAME = sft_checkpoint
print(f'SFT done. GRPO will start from: {sft_checkpoint}')


## 5. Train â€” ADAPT-focused live reward dashboard

Training script emits `[ATC_REWARD step=N]` lines every 20 reward calls.  
This cell streams stdout and shows a per-role dashboard in real time.

> **Expected:** ADAPT reward starts near 0.0 and climbs to 0.3â€“0.5 over 100 episodes.

In [ ]:
import time, subprocess, sys, os, re
from collections import defaultdict
from pathlib import Path

# â”€â”€ Defaults (safe fallback if config cell wasn't run) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_g = globals()
MODEL_NAME    = _g.get("MODEL_NAME",    "Qwen/Qwen2.5-1.5B-Instruct")
OUTPUT_DIR    = Path(_g.get("OUTPUT_DIR", "/content/atc-adapt"))
EPISODES      = _g.get("EPISODES",      100)
N_GENERATIONS = _g.get("N_GENERATIONS", 2)
LORA_RANK     = _g.get("LORA_RANK",     32)
SEED          = _g.get("SEED",          42)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODEL_NAME    = {MODEL_NAME}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")
print(f"EPISODES      = {EPISODES}  |  N_GENERATIONS = {N_GENERATIONS}  |  LORA_RANK = {LORA_RANK}")

# â”€â”€ Build command â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
train_cmd = [
    sys.executable, "training/train_grpo.py",
    "--model",         str(MODEL_NAME),
    "--output_dir",    str(OUTPUT_DIR),
    "--episodes",      str(EPISODES),
    "--n_generations", str(N_GENERATIONS),
    "--lora_rank",     str(LORA_RANK),
    "--seed",          str(SEED),
]

# â”€â”€ Live reward tracking (3 roles: ADAPT, AMAN, DMAN) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_ALL_ROLES     = ("ADAPT", "AMAN", "DMAN", "composite")
reward_history = defaultdict(list)
loss_history   = []
step_history   = []

_re_atc  = re.compile(r"\[ATC_REWARD step=(\d+)\] (.+)")
_re_kv   = re.compile(r"(\w+)=([\-\d.eE+]+)")
_re_loss = re.compile(r"'loss':\s*([\-\d.eE+]+)")

def _bar(val, width=24):
    frac   = max(0.0, min(1.0, (val + 1.0) / 2.0))   # map [-1,1] -> [0,1]
    filled = int(frac * width)
    sym    = "#" if val >= 0 else "-"
    return f"[{sym * filled}{' ' * (width - filled)}] {val:+.3f}"

def _trend(vals):
    if len(vals) < 40: return ""
    half  = len(vals) // 2
    d     = sum(vals[half:]) / max(1, len(vals)-half) - sum(vals[:half]) / half
    return "  UP" if d > 0.03 else ("  DN" if d < -0.03 else "  --")

def _dashboard(step, elapsed_s):
    print(f"\n{'='*56}")
    print(f"  step {step:5d}   elapsed {elapsed_s/60:.1f} min")
    print(f"{'='*56}")
    if loss_history:
        print(f"  {'loss':12s}: {loss_history[-1]:.5f}")
    for role in _ALL_ROLES:
        vals = reward_history.get(role, [])
        if not vals: continue
        m = sum(vals[-20:]) / min(20, len(vals))
        print(f"  {role:12s}: {_bar(m)}{_trend(vals)}")
    print()

# â”€â”€ Launch â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
env_copy = os.environ.copy()
env_copy["PYTHONUNBUFFERED"] = "1"

print("\nCommand:", " ".join(train_cmd))
print("=" * 60)
t0 = time.time()
last_dash = -1

proc = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env_copy,
)

for raw in proc.stdout:
    line = raw.rstrip("\n")
    print(line, flush=True)

    m_atc = _re_atc.match(line)
    if m_atc:
        step = int(m_atc.group(1))
        for kv in _re_kv.finditer(m_atc.group(2)):
            reward_history[kv.group(1)].append(float(kv.group(2)))
        if step != last_dash:
            last_dash = step
            step_history.append(step)
            _dashboard(step, time.time() - t0)
        continue

    if "'loss'" in line:
        m_l = _re_loss.search(line)
        if m_l: loss_history.append(float(m_l.group(1)))

proc.wait()
elapsed = time.time() - t0
print("=" * 60)

print("\n=== FINAL REWARD SUMMARY ===")
for role in _ALL_ROLES:
    vals = reward_history.get(role, [])
    if not vals: continue
    n = len(vals)   
    avg = sum(vals) / n
    fq  = sum(vals[:max(1,n//4)]) / max(1,n//4)
    lq  = sum(vals[max(0,3*n//4):]) / max(1, n - 3*n//4)
    tr  = "UP" if lq > fq + 0.03 else ("DN" if lq < fq - 0.03 else "--")
    print(f"  {role:12s}: mean={avg:+.3f}  Q1={fq:+.3f}  Q4={lq:+.3f}  {tr}")

ok = proc.returncode == 0
print(f"\nTraining {'complete' if ok else 'FAILED (exit '+str(proc.returncode)+')'} in {elapsed/60:.1f} min")

## 6. Plot reward curves â€” ADAPT + AMAN + DMAN

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

_g = globals()
OUTPUT_DIR = Path(_g.get("OUTPUT_DIR", "/content/atc-adapt"))
plots_dir  = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "reward_curves.json") as f:
    curves = json.load(f)

COLOURS = {
    "ADAPT":"#9C27B0", "AMAN":"#2196F3", "DMAN":"#4CAF50", "composite":"#212121",
}

def smooth(v, w=20):
    return [sum(v[max(0,i-w):i+1])/min(w,i+1) for i in range(len(v))]

# â”€â”€ Per-role subplots â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("ADAPT-Focused GRPO â€” Reward Curves (3 agents + composite)",
             fontsize=14, fontweight="bold")

for ax, role in zip(axes.flat, ["ADAPT", "AMAN", "DMAN", "composite"]):
    vals = curves.get(role, [])
    if not vals:
        ax.set_title(f"{role} (no data)"); continue
    col = COLOURS.get(role, "#607D8B")
    ax.plot(vals, alpha=0.2, color=col, lw=0.7)
    ax.plot(smooth(vals), color=col, lw=2.0, label="smoothed")
    tail = vals[-max(1,len(vals)//4):]
    fm   = sum(tail)/len(tail)
    ax.axhline(fm, color=col, lw=1.0, ls=":")
    ax.set_title(f"{role}  (final={fm:+.3f})", fontweight="bold", color=col)
    ax.set_xlabel("Training call")
    ax.set_ylabel("Reward")
    ax.set_ylim(-1.05, 1.05)
    ax.axhline(0, color="gray", lw=0.5, ls="--")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
p1 = plots_dir / "training_curves_adapt_focused.png"
plt.savefig(p1, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {p1}")

# â”€â”€ ADAPT progress (primary role) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
adapt_vals = curves.get("ADAPT", [])
if adapt_vals:
    fig2, ax2 = plt.subplots(figsize=(12, 4))
    ax2.plot(adapt_vals, alpha=0.2, color="#9C27B0", lw=0.7)
    ax2.plot(smooth(adapt_vals, 30), color="#9C27B0", lw=2.2, label="smoothed (w=30)")
    q1e = max(1, len(adapt_vals)//4);  q4s = max(0, 3*len(adapt_vals)//4)
    q1m = sum(adapt_vals[:q1e])/q1e;   q4m = sum(adapt_vals[q4s:])/max(1,len(adapt_vals)-q4s)
    ax2.axhline(q1m, color="red",   lw=1.2, ls="--", label=f"Q1={q1m:+.3f}")
    ax2.axhline(q4m, color="green", lw=1.2, ls="--", label=f"Q4={q4m:+.3f}")
    ax2.fill_between(range(len(adapt_vals)), q1m, q4m, alpha=0.07,
                     color="green" if q4m > q1m else "red")
    ax2.set_title("ADAPT Reward â€” Domain Transfer Learning Progress", fontweight="bold")
    ax2.set_xlabel("Training call")
    ax2.set_ylabel("ADAPT reward")
    ax2.set_ylim(-1.05, 1.05)
    ax2.axhline(0, color="gray", lw=0.5, ls=":")
    ax2.legend(); ax2.grid(True, alpha=0.3)
    p2 = plots_dir / "adapt_progress.png"
    plt.tight_layout()
    plt.savefig(p2, dpi=150, bbox_inches="tight")
    plt.show()
    d = q4m - q1m
    print(f"Saved: {p2}")
    print(f"Q1->Q4 delta: {d:+.3f}  ({'learning' if d>0.03 else 'stable' if d>-0.03 else 'degraded'})")

## 7. Before / After comparison

In [ ]:
import json, matplotlib.pyplot as plt, numpy as np
from pathlib import Path

_g = globals()
OUTPUT_DIR = Path(_g.get("OUTPUT_DIR", "/content/atc-adapt"))
plots_dir  = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

bp = OUTPUT_DIR / "baseline_metrics.json"
tp = OUTPUT_DIR / "trained_model_metrics.json"

if not bp.exists() or not tp.exists():
    print("Metrics files not found â€” training runs eval automatically unless --no_eval.")
else:
    with open(bp) as f: baseline = json.load(f)
    with open(tp) as f: trained  = json.load(f)

    metrics = [
        ("mean_composite",   "Composite score"),
        ("mean_aman_reward", "AMAN reward"),
        ("mean_dman_reward", "DMAN reward"),
        ("mean_conflicts",   "Avg conflicts (lower)"),
        ("mean_emg_handled", "Emergencies handled"),
    ]
    labels = [m[1] for m in metrics]
    before = [baseline.get(m[0], 0) for m in metrics]
    after  = [trained.get(m[0], 0)  for m in metrics]

    x, w = np.arange(len(labels)), 0.35
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x-w/2, before, w, label=baseline.get("tag","Baseline"), color="#90A4AE")
    ax.bar(x+w/2, after,  w, label=trained.get("tag", "Trained"),  color="#9C27B0")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("Before vs. After GRPO â€” ADAPT-Focused Training", fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    for b, a, xi in zip(before, after, x):
        d = a - b
        ax.text(xi+w/2, max(a,b)+0.02, f"{d:+.3f}", ha="center", fontsize=9,
                color="green" if d > 0 else "red")
    plt.tight_layout()
    p3 = plots_dir / "before_after.png"
    plt.savefig(p3, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {p3}")

    print("\n=== BEFORE vs AFTER ===")
    for key, label in metrics:
        b, a = baseline.get(key,0), trained.get(key,0)
        d    = a - b
        arr  = "UP" if d > 0.005 else ("DN" if d < -0.005 else "--")
        print(f"  {label:34s}: {b:6.3f}  ->  {a:6.3f}  ({d:+.3f} {arr})")

## Troubleshooting

| Problem | Fix |
|---|---|
| OOM on T4 with 7B model | Use `Qwen/Qwen2.5-1.5B-Instruct` — default is already 1.5B. |
| `get_full_repo_name` ImportError | Re-run install cell — `trl<0.16` broken with `huggingface_hub>=0.24`. |
| SFT `formatting_func` error | Notebook pre-processes to `text` column — re-run SFT cell. |
| ADAPT reward stuck at ~0.0 | Check rationale quality. Ensure domain_episode_ratio=0.65. |
| Parse failures dominating | Expected early on. Soft penalty (-0.2) allows gradual learning. |
| `GRPOConfig` keyword error | Re-run install cell â€” needs `trl==0.15.2`. |
| Colab disconnects | Keep `OUTPUT_DIR` on Drive. Reward curves saved after each step. |
| `torch._dynamo` errors | Normal with Unsloth â€” compatibility shims handle this. |